In [1]:
import numpy as np

# Neural Network Structure

* in (5,10) 
* int (10,100)
* out (100,1)


In [70]:
import matplotlib.pyplot as plt

# generate fake data to train on
def groud_truth(x):
    res = np.sin(x)
    res = res * 1/(1+np.exp(x))
    res = np.sum(res, axis = -1)
    return res[:, None]

x_test = np.random.rand(batch,5)
groud_truth(x_test).shape

(32, 1)

In [113]:
# batchsize 
BATCH = 32
# initalize weights
weights ={'l1': np.random.rand(5,10) , 'l2':np.random.rand(10,100), 'l3': np.random.rand(100,1)}      
bias = {'l1':np.random.rand(10)  , 'l2':np.random.rand(100) , 'l3': np.random.rand(1)}

# activation function
def relu(z):
    ind = np.where(z < 0)
    z[ind] = 0
    return z


def forward(x, weights, bias, activation):
    depth = len(weights.keys())
    for k in weights.keys():
        W, b = weights[k], bias[k]
        z = x@W+b
        x = activation(z)
    return x

x_test = np.random.rand(BATCH,5)

result = forward(x_test, weights, bias, relu)
print(result.shape)

(32, 1)


In [114]:
def loss(pred, true):
    return np.sum((pred-true)**2)

x_test = np.random.rand(batch,5)
true = groud_truth(x_test)
loss(forward(x_test, weights, bias, relu), true)

np.float64(7790999.523345705)

# Backprop

let us backprop the basic gradients

$$f_{\theta}(x) = \text{ReLU}(xW^T+b)$$
Apply Chain Rule
$$\nabla_W f_\theta(x) = d\text{ReLU}(xW^T+b)(x)^T $$
$$\nabla_b f_\theta(x) = d\text{ReLU}(xW^T+b)\mathbb{1}.T$$


In [191]:
def d_relu(z):
    ind_one = np.where(z>=0)
    grad = np.zeros(z.shape)
    grad[ind_one] = 1
    return grad
    
def d_loss(pred, true):
    return 2*(pred-true)

def forward(x, weights, bias, activation):
    depth = len(weights.keys())
    for k in weights.keys():
        W, b = weights[k], bias[k]
        z = x@W+b
        x_new = activation(z)
        x = x_new
    return x
    
result = forward(x_test, weights, bias, relu)


In [308]:
def grad_layer(d_activation, W, b, x_prev):
    z = x_prev@W+b
    W_expand = np.copy(W[None,:])
    W_expand = np.repeat(W_expand, BATCH, axis = 0)
    d_act = d_activation(z)
    d_act = d_act[:, None, :]
    x_prev = x_prev[:,None, :]
    dW = np.matmul(d_act.mT ,x_prev)
    ones = np.ones(b.shape)
    db = np.multiply(d_activation(z), ones)
    return dW,db

In [364]:
import numpy as np

def backwards(x, y, weights, bias, activation, d_activation, d_loss):
    layer_names = ['l1', 'l2', 'l3']
    
    # --- 1. FORWARD PASS ---
    x_l = {}
    z_l = {}
    
    current_x = x
    for k in layer_names:
        W, b = weights[k], bias[k]
        z = np.matmul(current_x, W) + b
        
        x_l[k] = current_x      # Save input to the layer (X_prev)
        z_l[k] = z              # Save pre-activation (Z)
        
        current_x = activation(z) # Output of the layer

    # --- 2. BACKWARD PASS ---
    grads_W = {}
    grads_b = {}
    layer_names_reverse = ['l3', 'l2', 'l1']

    dA = d_loss(current_x, y) 
    l = loss(current_x, y) 
    for k in layer_names_reverse:
        # a. Element-wise multiply upstream gradient by activation derivative
        dZ = dA * d_activation(z_l[k])
        
        # b. Calculate gradients for Weights and Biases
        # x_l[k].T @ dZ naturally sums the gradients over the batch dimension
        grads_W[k] = np.matmul(x_l[k].T, dZ) 
        grads_b[k] = np.sum(dZ, axis=0, keepdims=True) # Sum across batch
        
        # c. Calculate the gradient to pass to the previous layer
        dA = np.matmul(dZ, weights[k].T) 
        
    return grads_W, grads_b, l

grads_W, grads_b, l = backwards(x_test, true, weights, bias, relu, d_relu, d_loss)


In [329]:
# def backwards(x, y, weights, bias, activation):
#     layer_names = ['l1', 'l2', 'l3']
#     depth = len(weights.keys())
#     x_l = {}
#     for k in layer_names:
#         W, b = weights[k], bias[k]
#         z = x@W+b
#         x_l[k] = np.copy(x)
#         x_new = activation(z)
#         x = x_new

#     grads_W = {}
#     grads_b = {}
#     layer_names_reverse = ['l3', 'l2', 'l1']
#     dloss = d_loss(x, true)[:,None]
#     for i in range(len(layer_names_reverse)):
#         k = layer_names_reverse[i]
#         if i>0:
#             k_prev = layer_names_reverse[i-1]
#         dW_temp, db_temp = grad_layer(d_relu, 
#                                       weights[k], bias[k], 
#                                       x_l[k])
#         db_temp = db_temp[:,:,None]
#         if k == 'l3':

#             grads_W[k] = np.matmul(dloss, dW_temp)
#             grads_b[k] = np.matmul(dloss, db_temp)
#         else:

            
#             # print('new', dW_temp.mT.shape, 'prev', grads_W[k_prev].shape)
#             grads_W[k] = np.matmul(dW_temp.mT, grads_W[k_prev].mT).mT
#             # print(db_temp.shape, grads_b[k_prev].shape)
#             grads_b[k] = np.matmul(db_temp, grads_b[k_prev]).mT
#     return grads_W, grads_b

# grads_W, grads_b = backwards(x_test, true, weights, bias, relu)

In [363]:
# batchsize 
BATCH = 32
lr = 0.0001
# initalize weights
# He Initialization: mean 0, variance 2/input_size
weights = {
    'l1': np.random.randn(5, 10) * np.sqrt(2. / 5), 
    'l2': np.random.randn(10, 100) * np.sqrt(2. / 10), 
    'l3': np.random.randn(100, 1) * np.sqrt(2. / 100)
}      

# Initialize biases to 0
bias = {
    'l1': np.zeros(10), 
    'l2': np.zeros(100), 
    'l3': np.zeros(1)
}

def update(lr, W, b, dW, db):
    for k in W.keys():
        W[k] -= lr * dW[k]
        b[k] -= lr * db[k][0,:]
    return W,b

for i in range(100):
    grads_W, grads_b, l = backwards(x_test, true, weights, bias, relu, d_relu, d_loss)
    weights, bias = update(lr, weights, bias, grads_W, grads_b)
    print(l)

11.47445915881729
9.304761318106959
7.639210229446794
6.320866215648037
5.29669819366466
4.514312822292637
3.9049322320966384
3.417613125904452
3.0438361767450552
2.756367559711515
2.533165645014691
2.3591514037287222
2.223216934060003
2.1161606031331157
2.0316018368223827
1.9642493280432334
1.9101003537651897
1.8661693107750792
1.830092226727963
1.8000194866917616
1.7745063754095445
1.7525364260087044
1.733302325213152
1.7161962252648773
1.7007337013016701
1.686541868752542
1.6733280858622357
1.6609031852732536
1.64910210312964
1.6377986029875369
1.6267261497601402
1.615992096726061
1.605540240396772
1.5953263227877441
1.5853165555015587
1.5754851457538104
1.5658123984364505
1.5562832611156199
1.5468862095350588
1.5376123947969644
1.5284549915558467
1.5194085503566392
1.5104682468751036
1.5016314148527634
1.492895389607623
1.4842579866495886
1.4757197882190711
1.4672864544250235
1.4589466348606202
1.4507234181579387
1.4425941104685265
1.4345537519555573
1.426601356324231
1.41873610518

# Stochastic

In [408]:
weights = {
    'l1': np.random.randn(5, 10) * np.sqrt(2. / 5), 
    'l2': np.random.randn(10, 100) * np.sqrt(2. / 10), 
    'l3': np.random.randn(100, 1) * np.sqrt(2. / 100)
}      

# Initialize biases to 0
bias = {
    'l1': np.zeros(10), 
    'l2': np.zeros(100), 
    'l3': np.zeros(1)
}

rng = np.random.default_rng()
for i in range(100):
    mini_batch = rng.choice(32, size=1, replace=False)
    grads_W, grads_b, l = backwards(x_test[mini_batch,:], true[mini_batch, :], weights, bias, relu, d_relu, d_loss)
    for k in weights.keys():
        weights[k] -= lr * grads_W[k]
        bias[k] -= lr * grads_b[k][0,:]
    if i % 10==0:    
        print(l)

0.8789380582506959
0.3831616993624187
0.03672477530631556
0.2731484872732361
0.8147133976885503
0.6798629881113005
0.55939187497371
0.24132454931580158
0.260496499692568
0.05878627075512433


In [421]:
from copy import copy
weights = {
    'l1': np.random.randn(5, 10) * np.sqrt(2. / 5), 
    'l2': np.random.randn(10, 100) * np.sqrt(2. / 10), 
    'l3': np.random.randn(100, 1) * np.sqrt(2. / 100)
}      

# Initialize biases to 0
bias = {
    'l1': np.zeros(10), 
    'l2': np.zeros(100), 
    'l3': np.zeros(1)
}

rng = np.random.default_rng()
beta = 0.01
eps = 1e-8

v_W = {k: np.zeros_like(v) for k, v in weights.items()}
v_b = {k: np.zeros_like(v) for k, v in bias.items()}

for i in range(100):
    mini_batch = rng.choice(32, size=1, replace=False)
    
    grads_W, grads_b, l = backwards(x_test[mini_batch,:], true[mini_batch, :], weights, bias, relu, d_relu, d_loss)
    if i ==0:
        grads_W_prev, grads_b_prev = copy(grads_W), copy(grads_b)
        
    for k in weights.keys():
        v_W[k] = beta * v_W[k] + (1-beta)*grads_W[k] 
        v_b[k] = beta * v_b[k] + (1-beta)*grads_b[k] 

        weights[k] -= lr * v_W[k]
        bias[k] -= lr * v_b[k][0,:]

        grads_W_prev, grads_b_prev = copy(grads_W), copy(grads_b)
    if i % 10==0:    
        print(l)

0.6999454222015496
0.5146615680212933
0.5184704741600863
0.48449261957106615
0.44183718355429896
0.9003773094229198
1.1586769377328956
0.25138116083421025
0.4290618242149764
0.3775632273356969


In [435]:
weights = {
    'l1': np.random.randn(5, 10) * np.sqrt(2. / 5), 
    'l2': np.random.randn(10, 100) * np.sqrt(2. / 10), 
    'l3': np.random.randn(100, 1) * np.sqrt(2. / 100)
}      

# Initialize biases to 0
bias = {
    'l1': np.zeros(10), 
    'l2': np.zeros(100), 
    'l3': np.zeros(1)
}

S_W = {k: np.zeros_like(v) for k, v in weights.items()}
S_b = {k: np.zeros_like(v) for k, v in bias.items()}

for i in range(100):
    mini_batch = rng.choice(32, size=1, replace=False)
    
    grads_W, grads_b, l = backwards(x_test[mini_batch,:], true[mini_batch, :], weights, bias, relu, d_relu, d_loss)
    if i ==0:
        grads_W_prev, grads_b_prev = copy(grads_W), copy(grads_b)
        
    for k in weights.keys():
        S_W[k] = S_W[k] + grads_W[k] **2
        S_b[k] = S_b[k] + grads_b[k] **2
        
        weights[k] -= lr/(np.sqrt(S_W[k]) + eps) * grads_W[k]
        bias[k] -= lr/(np.sqrt(S_b[k][0,:]) + eps) * grads_b[k][0,:]

        grads_W_prev, grads_b_prev = copy(grads_W), copy(grads_b)
    if i % 10==0:    
        print(l)

0.01915527894946685
0.3295832407348076
0.530434461521226
0.0589612857055966
0.19500514134813401
0.05079804936627793
0.11850265346558472
0.010306340546840811
0.18737233407929615
0.021936551317234827


In [ ]:
weights = {
    'l1': np.random.randn(5, 10) * np.sqrt(2. / 5), 
    'l2': np.random.randn(10, 100) * np.sqrt(2. / 10), 
    'l3': np.random.randn(100, 1) * np.sqrt(2. / 100)
}      

# Initialize biases to 0
bias = {
    'l1': np.zeros(10), 
    'l2': np.zeros(100), 
    'l3': np.zeros(1)
}

S_W = {k: np.zeros_like(v) for k, v in weights.items()}
S_b = {k: np.zeros_like(v) for k, v in bias.items()}
v_W = {k: np.zeros_like(v) for k, v in weights.items()}
v_b = {k: np.zeros_like(v) for k, v in bias.items()}

for i in range(100):
    mini_batch = rng.choice(32, size=1, replace=False)
    
    grads_W, grads_b, l = backwards(x_test[mini_batch,:], true[mini_batch, :], weights, bias, relu, d_relu, d_loss)
    if i ==0:
        grads_W_prev, grads_b_prev = copy(grads_W), copy(grads_b)
        
    for k in weights.keys():
        S_W[k] = S_W[k] + grads_W[k] **2
        S_b[k] = S_b[k] + grads_b[k] **2
        
        weights[k] -= lr/(np.sqrt(S_W[k]) + eps) * grads_W[k]
        bias[k] -= lr/(np.sqrt(S_b[k][0,:]) + eps) * grads_b[k][0,:]

        grads_W_prev, grads_b_prev = copy(grads_W), copy(grads_b)
    if i % 10==0:    
        print(l)

In [444]:
weights = {
    'l1': np.random.randn(5, 10) * np.sqrt(2. / 5), 
    'l2': np.random.randn(10, 100) * np.sqrt(2. / 10), 
    'l3': np.random.randn(100, 1) * np.sqrt(2. / 100)
}      

# Initialize biases to 0
bias = {
    'l1': np.zeros(10), 
    'l2': np.zeros(100), 
    'l3': np.zeros(1)
}

S_W = {k: np.zeros_like(v) for k, v in weights.items()}
S_b = {k: np.zeros_like(v) for k, v in bias.items()}
v_W = {k: np.zeros_like(v) for k, v in weights.items()}
v_b = {k: np.zeros_like(v) for k, v in bias.items()}
beta = 0.9  # NEW: Decay rate for the moving average
for i in range(100):
    mini_batch = rng.choice(32, size=1, replace=False)
    
    grads_W, grads_b, l = backwards(x_test[mini_batch,:], true[mini_batch, :], weights, bias, relu, d_relu, d_loss)
    if i ==0:
        grads_W_prev, grads_b_prev = copy(grads_W), copy(grads_b)
        
    for k in weights.keys():
        v_W[k] = beta * v_W[k] + (1-beta)*grads_W[k] 
        v_b[k] = beta * v_b[k] + (1-beta)*grads_b[k] 

        v_W[k] = v_W[k] /(1-beta)
        v_b[k] = v_b[k] /(1-beta)
        
        S_W[k] = beta * S_W[k] + (1-beta) *grads_W[k] **2
        S_b[k] = beta * S_b[k] + (1-beta) *grads_b[k] **2

        S_W[k] = S_W[k]/(1-beta)
        S_b[k] = S_b[k]/(1-beta)
        
        weights[k] -= lr/(np.sqrt(S_W[k]) + eps) * v_W[k]
        bias[k] -= lr/(np.sqrt(S_b[k][0,:]) + eps) * v_b[k][0,:]

        grads_W_prev, grads_b_prev = copy(grads_W), copy(grads_b)
    if i % 10==0:    
        print(l)

0.027207498736814374
0.48449261957106615
0.7355943359882495
0.5653285753804531
0.9003773094229198
0.7355943359882495
0.5587683845970294
0.6704273796732755
0.7587648305738325
0.8147133976885503
